# Crear DataFrames - Premier League (últimas 10 temporadas)

Este notebook lee los ficheros `premierleague.txt` de cada subcarpeta dentro de `data/` y genera un Excel con un registro por partido con las siguientes variables:

- `Fecha` (DateTime, DD/MM/AAAA HH:MM)
- `EquipoLocal`
- `EquipoVisitante`
- `GolesMarcadosLocal`
- `GolesMarcadosVisitante`
- `GolesMarcadosDescansoLocal`
- `GolesMarcadosDescansoVisitante`

In [1]:
import os
import re
import pandas as pd
from datetime import datetime

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# FUNCIÓN PRINCIPAL DE PARSEO
# Soporta dos formatos de TXT que usa football-data.org / worldfootball.net:
#
#  Formato A (temporadas antiguas):
#    HH:MM  Equipo Local   G-G (G-G)  Equipo Visitante
#
#  Formato B (temporadas recientes):
#    HH:MM  Equipo Local   v  Equipo Visitante   G-G (G-G)
# ─────────────────────────────────────────────────────────────────────────────

# Regex para detectar línea de fecha (encabezado de día)
RE_DATE_HEADER = re.compile(
    r'^\s*(Mon|Tue|Wed|Thu|Fri|Sat|Sun)\s+'
    r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+'
    r'(\d{1,2})(?:\s+(\d{4}))?',
    re.IGNORECASE
)

# Regex para detectar hora al principio de línea de partido
RE_TIME = re.compile(r'^\s*(\d{1,2}:\d{2})\s+')

# Regex Formato B: "EquipoLocal  v  EquipoVisitante  G-G (G-G)"
RE_FORMAT_B = re.compile(
    r'^(?P<home>.+?)\s+v\s+(?P<away>.+?)\s+'
    r'(?P<hg>\d+)-(?P<ag>\d+)'
    r'(?:\s+\((?P<hhg>\d+)-(?P<hag>\d+)\))?\s*$'
)


def _parse_score_part(text: str):
    """
    Intenta extraer (hg, ag, hhg, hag) de un fragmento de texto
    que contiene "G-G" o "G-G (G-G)".
    Devuelve None si no hay marcador (partido no jugado).
    """
    m = re.search(
        r'(\d+)-(\d+)(?:\s*\((\d+)-(\d+)\))?',
        text
    )
    if not m:
        return None
    hg  = int(m.group(1))
    ag  = int(m.group(2))
    hhg = int(m.group(3)) if m.group(3) is not None else None
    hag = int(m.group(4)) if m.group(4) is not None else None
    return hg, ag, hhg, hag


def parse_season_txt(filepath: str) -> pd.DataFrame:
    """
    Parsea un fichero TXT de una temporada de la Premier League
    y devuelve un DataFrame con las columnas requeridas.
    """
    records = []

    # Leer las líneas del fichero
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        raw_lines = f.readlines()

    # Extraer el año de inicio de la temporada del encabezado (primera línea)
    # Ej: "= English Premier League 2024/25"  →  season_start_year = 2024
    season_start_year = None
    header_match = re.search(r'(\d{4})/(\d{2})', raw_lines[0] if raw_lines else '')
    if header_match:
        season_start_year = int(header_match.group(1))
    else:
        # fallback: intentar extraerlo del nombre de la carpeta
        folder = os.path.basename(os.path.dirname(filepath))
        y_match = re.match(r'(\d{4})', folder)
        if y_match:
            season_start_year = int(y_match.group(1))

    current_day    = None   # datetime.date del último encabezado de día visto
    last_time      = None   # string HH:MM de la última hora vista
    current_year   = season_start_year  # año activo
    last_month_num = None   # para detectar salto de año (dic → ene)

    MONTHS = {
        'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4,
        'may': 5, 'jun': 6, 'jul': 7, 'aug': 8,
        'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
    }

    for raw_line in raw_lines:
        line = raw_line.rstrip('\r\n')

        # ── Detectar encabezado de día ──────────────────────────────────────
        date_m = RE_DATE_HEADER.match(line)
        if date_m:
            month_str = date_m.group(2).lower()
            month_num = MONTHS.get(month_str)
            day_num   = int(date_m.group(3))
            year_str  = date_m.group(4)  # puede ser None

            if year_str:
                current_year = int(year_str)
            else:
                # Detectar salto de año automáticamente:
                # si el mes cae por debajo del mes anterior → nuevo año
                if last_month_num and month_num and month_num < last_month_num:
                    current_year += 1

            last_month_num = month_num

            try:
                current_day = datetime(current_year, month_num, day_num).date()
            except Exception:
                current_day = None

            last_time = None  # reset de hora al cambiar de día
            continue

        # ── Detectar hora al inicio de línea ─────────────────────────────
        time_m = RE_TIME.match(line)
        if time_m:
            last_time = time_m.group(1)
            # El resto de la línea es el partido
            rest = line[time_m.end():].strip()
        else:
            # Línea sin hora (mismo horario que la anterior)
            rest = line.strip()
            if not rest:
                continue

        # Ignorar líneas que no parecen un partido
        if current_day is None:
            continue
        if not rest:
            continue
        # Ignorar líneas de sección o comentario
        if rest.startswith('#') or rest.startswith('=') or 'Matchday' in rest:
            continue

        # ── Detectar formato ─────────────────────────────────────────────
        home = away = None
        hg = ag = hhg = hag = None

        # Intentar Formato B primero (tiene " v ")
        if ' v ' in rest:
            m_b = RE_FORMAT_B.match(rest)
            if m_b:
                home = m_b.group('home').strip()
                away = m_b.group('away').strip()
                hg   = int(m_b.group('hg'))
                ag   = int(m_b.group('ag'))
                hhg  = int(m_b.group('hhg')) if m_b.group('hhg') is not None else None
                hag  = int(m_b.group('hag')) if m_b.group('hag') is not None else None

        # Si no se parseó con Formato B, intentar Formato A
        if home is None:
            # El marcador está en medio: detectar posición del score
            score_m = re.search(r'(\d+)-(\d+)', rest)
            if score_m:
                score_start = score_m.start()

                home_part  = rest[:score_start].strip()
                score_part = rest[score_start:]

                parsed = _parse_score_part(score_part)
                if parsed:
                    hg, ag, hhg, hag = parsed

                    # La parte de visitante está después del marcador
                    after_score = re.sub(
                        r'^\d+-\d+(?:\s*\(\d+-\d+\))?',
                        '', score_part
                    ).strip()

                    home = home_part
                    away = after_score

        # Si no se pudo parsear, ignorar la línea
        if home is None or not home or not away:
            continue

        # ── Construir fecha-hora ────────────────────────────────────────────
        if last_time:
            try:
                h, mn = map(int, last_time.split(':'))
                dt = datetime(
                    current_day.year, current_day.month, current_day.day,
                    h, mn
                )
            except Exception:
                dt = datetime(
                    current_day.year, current_day.month, current_day.day
                )
        else:
            dt = datetime(
                current_day.year, current_day.month, current_day.day
            )

        records.append({
            'Fecha':                       dt,
            'EquipoLocal':                 home,
            'EquipoVisitante':             away,
            'GolesMarcadosLocal':          hg,
            'GolesMarcadosVisitante':      ag,
            'GolesMarcadosDescansoLocal':  hhg,
            'GolesMarcadosDescansoVisitante': hag,
        })

    df = pd.DataFrame(records, columns=[
        'Fecha',
        'EquipoLocal',
        'EquipoVisitante',
        'GolesMarcadosLocal',
        'GolesMarcadosVisitante',
        'GolesMarcadosDescansoLocal',
        'GolesMarcadosDescansoVisitante',
    ])

    # Convertir columnas de goles a tipo numérico entero (permite NaN con Int64)
    for col in [
        'GolesMarcadosLocal', 'GolesMarcadosVisitante',
        'GolesMarcadosDescansoLocal', 'GolesMarcadosDescansoVisitante'
    ]:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    return df


print("Funciones de parseo definidas correctamente.")

Funciones de parseo definidas correctamente.


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# BUCLE PRINCIPAL: procesar cada temporada y guardar Excel
# ─────────────────────────────────────────────────────────────────────────────

# Ruta de la carpeta data (relativa al notebook)
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'data')

season_dirs = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])

print(f"Temporadas encontradas: {season_dirs}\n")

for season in season_dirs:
    season_path = os.path.join(DATA_DIR, season)
    txt_file    = os.path.join(season_path, 'premierleague.txt')

    if not os.path.exists(txt_file):
        print(f"  [{season}] ⚠️  No se encontró premierleague.txt — se omite.")
        continue

    print(f"  [{season}] Parseando...", end=' ')

    df = parse_season_txt(txt_file)

    excel_path = os.path.join(season_path, f'premier_league_{season}.xlsx')
    df.to_excel(excel_path, index=False)

    print(f"{len(df)} partidos guardados → {os.path.basename(excel_path)}")

    # Mostrar las primeras filas de cada temporada como verificación
    display(df.head(3))
    print()

print("✅ Proceso completado. Se han generado los Excels en cada subcarpeta de data/.")

Temporadas encontradas: ['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26', '2026-27']

  [2016-17] Parseando... 380 partidos guardados → premier_league_2016-17.xlsx


,Fecha,EquipoLocal,EquipoVisitante,GolesMarcadosLocal,GolesMarcadosVisitante,GolesMarcadosDescansoLocal,GolesMarcadosDescansoVisitante
0,2016-08-13 12:30:00,Hull City,Leicester City,2,1,1,0
1,2016-08-13 15:00:00,Burnley FC,Swansea City,0,1,0,0
2,2016-08-13 15:00:00,Crystal Palace,West Bromwich Albion,0,1,0,0



  [2017-18] Parseando... 380 partidos guardados → premier_league_2017-18.xlsx


,Fecha,EquipoLocal,EquipoVisitante,GolesMarcadosLocal,GolesMarcadosVisitante,GolesMarcadosDescansoLocal,GolesMarcadosDescansoVisitante
0,2017-08-11 19:45:00,Arsenal FC,Leicester City,4,3,2,2
1,2017-08-12 12:30:00,Watford FC,Liverpool FC,3,3,2,1
2,2017-08-12 15:00:00,Chelsea FC,Burnley FC,2,3,0,3



  [2018-19] Parseando... 380 partidos guardados → premier_league_2018-19.xlsx


...


  [2025-26] Parseando... 380 partidos guardados → premier_league_2025-26.xlsx


,Fecha,EquipoLocal,EquipoVisitante,GolesMarcadosLocal,GolesMarcadosVisitante,GolesMarcadosDescansoLocal,GolesMarcadosDescansoVisitante
0,2025-08-15 19:00:00,Liverpool,Bournemouth,4,2,1,0
1,2025-08-16 12:30:00,Aston Villa,Newcastle United,0,0,0,0
2,2025-08-16 14:00:00,Sunderland,West Ham United,3,0,0,0



  [2026-27] Parseando... 10 partidos guardados → premier_league_2026-27.xlsx


,Fecha,EquipoLocal,EquipoVisitante,GolesMarcadosLocal,GolesMarcadosVisitante,GolesMarcadosDescansoLocal,GolesMarcadosDescansoVisitante
0,2026-08-21 20:00:00,Arsenal FC,Coventry City FC,3,0,2,0
1,2026-08-22 12:30:00,Hull City AFC,Manchester United FC,2,0,2,0
2,2026-08-22 15:00:00,Ipswich Town FC,Sunderland AFC,2,1,1,1



✅ Proceso completado. Se han generado los Excels en cada subcarpeta de data/.


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# VERIFICACIÓN OPCIONAL: cargar y mostrar el Excel de una temporada concreta
# ─────────────────────────────────────────────────────────────────────────────

SEASON_CHECK = '2026-27'  # Cambia aquí la temporada que quieras revisar

excel_check = os.path.join(DATA_DIR, SEASON_CHECK, f'premier_league_{SEASON_CHECK}.xlsx')
df_check = pd.read_excel(excel_check)

print(f"Temporada {SEASON_CHECK}: {len(df_check)} partidos")
print(f"Columnas: {list(df_check.columns)}")
print()
display(df_check)

Temporada 2026-27: 10 partidos
Columnas: ['Fecha', 'EquipoLocal', 'EquipoVisitante', 'GolesMarcadosLocal', 'GolesMarcadosVisitante', 'GolesMarcadosDescansoLocal', 'GolesMarcadosDescansoVisitante']



,Fecha,EquipoLocal,EquipoVisitante,GolesMarcadosLocal,GolesMarcadosVisitante,GolesMarcadosDescansoLocal,GolesMarcadosDescansoVisitante
0,2026-08-21 20:00:00,Arsenal FC,Coventry City FC,3,0,2,0
1,2026-08-22 12:30:00,Hull City AFC,Manchester United FC,2,0,2,0
2,2026-08-22 15:00:00,Ipswich Town FC,Sunderland AFC,2,1,1,1
3,2026-08-22 15:00:00,Nottingham Forest FC,Leeds United FC,0,1,0,0
4,2026-08-22 15:00:00,Everton FC,Crystal Palace FC,2,0,1,0
5,2026-08-22 17:30:00,Brentford FC,Tottenham Hotspur FC,3,0,2,0
6,2026-08-23 14:00:00,Manchester City FC,AFC Bournemouth,2,1,0,1
7,2026-08-23 14:00:00,Brighton & Hove Albion FC,Aston Villa FC,4,0,4,0
8,2026-08-23 16:30:00,Newcastle United FC,Liverpool FC,2,2,1,0
9,2026-08-24 20:00:00,Fulham FC,Chelsea FC,2,3,1,2
